# Purpose And Safety Boundary

This private Kaggle notebook validates a source-only, provider-free synthetic safety fixture on CPU. It uses no patient data, clinical dataset, model weights, provider API, internet, GPU, inference, or training.

In [ ]:
# Embedded Source Package Extraction
import base64
import hashlib
from io import BytesIO
import json
from pathlib import Path, PurePosixPath
import sys
import zipfile

SOURCE_ARCHIVE_SHA256 = "cd0c46ad8fab08736d3a62683271c96243c157987fa8aeee64db4f6f7027114b"
SOURCE_FILE_COUNT = 50
SOURCE_ARCHIVE_B64 = (
    "..."
    "+A4AAGmSAAAAAA=="
)
WORK_ROOT = Path("/kaggle/working/medllm-safety-public")
sys.dont_write_bytecode = True
if WORK_ROOT.exists():
    raise RuntimeError("Dedicated extraction directory already exists; restart the kernel before rerunning.")
archive_bytes = base64.b64decode(SOURCE_ARCHIVE_B64, validate=True)
if hashlib.sha256(archive_bytes).hexdigest() != SOURCE_ARCHIVE_SHA256:
    raise RuntimeError("Embedded source archive hash mismatch.")
with zipfile.ZipFile(BytesIO(archive_bytes)) as archive:
    for member in archive.infolist():
        candidate = PurePosixPath(member.filename)
        if candidate.is_absolute() or ".." in candidate.parts:
            raise RuntimeError("Unsafe embedded archive path.")
    archive.extractall(WORK_ROOT)
print(json.dumps({"source_files": SOURCE_FILE_COUNT, "work_root": str(WORK_ROOT), "archive_sha256": SOURCE_ARCHIVE_SHA256}, indent=2))

In [ ]:
# Source-Tree Validation
sys.path.insert(0, str(WORK_ROOT))
from medllm_safety.synthetic_validation import validate_source_tree

source_errors = validate_source_tree(WORK_ROOT)
print(json.dumps({"status": "passed" if not source_errors else "failed", "errors": source_errors}, indent=2))

In [ ]:
# Restricted-Artifact Scan
from medllm_safety.evidence import find_restricted_artifacts

restricted = find_restricted_artifacts(WORK_ROOT)
print(json.dumps({"restricted_artifact_count": len(restricted), "rules": sorted({item.rule for item in restricted})}, indent=2))

In [ ]:
# Environment And Version Inspection
from medllm_safety.synthetic_validation import dependency_versions
import platform

versions = dependency_versions()
environment_summary = {"dependency_versions": versions, "device": "cpu", "platform": platform.platform(), "gpu_used": False}
print(json.dumps(environment_summary, indent=2, sort_keys=True))

In [ ]:
# Compile Check
import os
import subprocess

command_environment = os.environ.copy()
command_environment["PYTHONDONTWRITEBYTECODE"] = "1"
command_environment["PYTHONPYCACHEPREFIX"] = "/kaggle/working/medllm-safety-pycache"
compile_process = subprocess.run(
    [sys.executable, "-m", "compileall", "-q", "medllm_safety", "project_preflight", "tests"],
    cwd=WORK_ROOT,
    env=command_environment,
    capture_output=True,
    text=True,
    timeout=120,
    check=False,
)
print(json.dumps({"step": "compile_check", "exit_code": compile_process.returncode}, indent=2))

In [ ]:
# Synthetic Test Suite
from medllm_safety.synthetic_validation import parse_pytest_count

test_process = subprocess.run(
    [sys.executable, "-m", "pytest", "-p", "no:cacheprovider"],
    cwd=WORK_ROOT,
    env=command_environment,
    capture_output=True,
    text=True,
    timeout=120,
    check=False,
)
test_output = "\n".join(part.strip() for part in (test_process.stdout, test_process.stderr) if part.strip())
print(test_output)
try:
    test_count = parse_pytest_count(test_output)
except ValueError:
    test_count = 0

In [ ]:
# Bounded Safety-Fixture Smoke Test
from medllm_safety.synthetic_validation import run_bounded_policy_smoke

smoke = run_bounded_policy_smoke()
print(json.dumps(smoke, indent=2, sort_keys=True))

In [ ]:
# Sanitized Evidence JSON Creation
from medllm_safety.evidence import build_synthetic_evidence, validate_evidence_manifest

validation_exit_code = int(bool(source_errors or restricted or compile_process.returncode or test_process.returncode))
evidence_relative_path = "synthetic-validation-evidence.json"
evidence_output = Path("/kaggle/working") / evidence_relative_path
configuration = {
    "mode": "provider_free_synthetic_validation",
    "network": False,
    "gpu": False,
    "provider_api_usage": "none",
    "model_inference": "not_run",
    "seed": 17,
    "timeout_seconds": 120,
}
evidence = build_synthetic_evidence(
    source_root=WORK_ROOT,
    configuration=configuration,
    dependency_versions=versions,
    seed=17,
    synthetic_sample_count=smoke["synthetic_sample_count"],
    test_count=test_count,
    exit_code=validation_exit_code,
    evidence_path=evidence_relative_path,
    restricted_artifact_count=len(restricted),
)
evidence.update({
    "source_tree_validation": "passed" if not source_errors else "failed",
    "compile_check": "passed" if compile_process.returncode == 0 else "failed",
    "synthetic_test_suite": "passed" if test_process.returncode == 0 else "failed",
    "bounded_policy_smoke": smoke,
    "stop_gate": "reached",
    "warnings": [],
})
validate_evidence_manifest(evidence)
evidence_output.write_text(json.dumps(evidence, indent=2, sort_keys=True) + "\n", encoding="utf-8")
print(json.dumps({"evidence_path": evidence_relative_path, "exit_code": validation_exit_code}, indent=2))

In [ ]:
# Evidence JSON Inspection
loaded_evidence = json.loads(evidence_output.read_text(encoding="utf-8"))
validate_evidence_manifest(loaded_evidence)
if loaded_evidence != evidence:
    raise RuntimeError("Evidence JSON round-trip mismatch.")
print(json.dumps(loaded_evidence, indent=2, sort_keys=True))

In [ ]:
# Final Bounded Result
if validation_exit_code != 0:
    raise RuntimeError("Bounded synthetic validation failed; inspect the sanitized evidence and first failing step.")
print("SYNTHETIC VALIDATION PASSED - APPROVAL GATE REACHED")

## APPROVAL GATE

Stop here. This notebook does not contain model/data access, provider calls, inference, GPU work, deployment, or clinical evaluation. No clinical or real-world safety conclusion is supported.